In [0]:
import pandas as pd
from datetime import datetime

CURRENT_YEAR = datetime.now().year

RISK_MODEL_VERSION = "v1.0"
STUDY_AREA_NAME = "Western Los Angeles County"

SILVER_STATIC_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_static_features"
)

SILVER_WEATHER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_weather"
)

SILVER_RISK_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_risk"
)

print("Current year:", CURRENT_YEAR)
print("Risk model:", RISK_MODEL_VERSION)

In [0]:
static_features = (
    spark.read
    .format("delta")
    .load(SILVER_STATIC_PATH)
    .toPandas()
)

weather = (
    spark.read
    .format("delta")
    .load(SILVER_WEATHER_PATH)
    .toPandas()
)

print("Static rows:", len(static_features))
print("Weather rows:", len(weather))

In [0]:
SILVER_STATIC_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_static_features"
)

SILVER_WEATHER_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_weather"
)

static_features = (
    spark.read
    .format("delta")
    .load(SILVER_STATIC_PATH)
    .toPandas()
)

weather = (
    spark.read
    .format("delta")
    .load(SILVER_WEATHER_PATH)
    .toPandas()
)

print("Static rows:", len(static_features))
print("Weather rows:", len(weather))

print(
    weather[
        ["pipeline_run_timestamp"]
    ].drop_duplicates()
)

In [0]:
zone_features = static_features.merge(
    weather,
    on="zone_id",
    how="left",
    validate="one_to_one"
)

print("Merged rows:", len(zone_features))
print(
    "Unique zones:",
    zone_features["zone_id"].nunique()
)
print(
    zone_features[
        ["pipeline_run_timestamp"]
    ].drop_duplicates()
)

In [0]:
zone_features["hazard_factor"] = (
    zone_features["hazard_score"] / 3
)

In [0]:
def get_wind_factor(gust):

    if gust < 15:
        return 0.0

    elif gust < 25:
        return 0.25

    elif gust < 40:
        return 0.50

    elif gust < 55:
        return 0.75

    else:
        return 1.0


zone_features["wind_factor"] = (
    zone_features["wind_gust_mph"]
    .apply(get_wind_factor)
)

In [0]:
def get_humidity_factor(humidity):

    if humidity >= 40:
        return 0.0

    elif humidity >= 30:
        return 0.25

    elif humidity >= 20:
        return 0.50

    elif humidity >= 10:
        return 0.75

    else:
        return 1.0


zone_features["humidity_factor"] = (
    zone_features["relative_humidity_pct"]
    .apply(get_humidity_factor)
)

In [0]:
def get_temperature_factor(temp):

    if temp < 75:
        return 0.0

    elif temp < 85:
        return 0.25

    elif temp < 95:
        return 0.50

    elif temp < 105:
        return 0.75

    else:
        return 1.0


zone_features["temperature_factor"] = (
    zone_features["temperature_f"]
    .apply(get_temperature_factor)
)

In [0]:
zone_features["years_since_last_fire"] = (
    CURRENT_YEAR
    - zone_features["last_fire_year"]
)


def get_history_factor(
    fire_count,
    years_since_fire
):

    if fire_count == 0:
        return 0.0

    count_factor = min(
        fire_count / 5,
        1.0
    )

    if years_since_fire <= 2:
        recency_factor = 1.0

    elif years_since_fire <= 5:
        recency_factor = 0.75

    elif years_since_fire <= 10:
        recency_factor = 0.50

    else:
        recency_factor = 0.25

    return (
        count_factor * 0.5
        +
        recency_factor * 0.5
    )


zone_features["history_factor"] = (
    zone_features.apply(
        lambda row:
        get_history_factor(
            row["historical_fire_count"],
            row["years_since_last_fire"]
        ),
        axis=1
    )
)

In [0]:
zone_features["hazard_contribution"] = (
    zone_features["hazard_factor"] * 40
)

zone_features["wind_contribution"] = (
    zone_features["wind_factor"] * 25
)

zone_features["humidity_contribution"] = (
    zone_features["humidity_factor"] * 20
)

zone_features["temperature_contribution"] = (
    zone_features["temperature_factor"] * 10
)

zone_features["history_contribution"] = (
    zone_features["history_factor"] * 5
)

In [0]:
zone_features["risk_score"] = (
    zone_features["hazard_contribution"]
    +
    zone_features["wind_contribution"]
    +
    zone_features["humidity_contribution"]
    +
    zone_features["temperature_contribution"]
    +
    zone_features["history_contribution"]
)

zone_features["risk_score"] = (
    zone_features["risk_score"]
    .round(1)
)

In [0]:
def get_risk_category(score):

    if score < 25:
        return "Low"

    elif score < 50:
        return "Moderate"

    elif score < 75:
        return "High"

    else:
        return "Critical"


zone_features["risk_category"] = (
    zone_features["risk_score"]
    .apply(get_risk_category)
)

zone_features["risk_model_version"] = (
    RISK_MODEL_VERSION
)

zone_features["study_area"] = (
    STUDY_AREA_NAME
)

In [0]:
display(
    zone_features[
        [
            "zone_id",
            "hazard_score",
            "temperature_f",
            "relative_humidity_pct",
            "wind_gust_mph",
            "historical_fire_count",
            "risk_score",
            "risk_category"
        ]
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(15)
)

In [0]:
print("Risk score range:")
print(
    zone_features["risk_score"].min(),
    "to",
    zone_features["risk_score"].max()
)

print("\nRisk categories:")
print(
    zone_features[
        "risk_category"
    ].value_counts()
)

print(
    "\nMissing risk scores:",
    zone_features[
        "risk_score"
    ].isna().sum()
)

print(
    "Scores outside 0-100:",
    (
        ~zone_features[
            "risk_score"
        ].between(0, 100)
    ).sum()
)

In [0]:
risk_spark_df = spark.createDataFrame(
    zone_features
)

(
    risk_spark_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_RISK_PATH)
)

print("Current risk dataset written successfully.")
print("Path:", SILVER_RISK_PATH)

In [0]:
check_risk = (
    spark.read
    .format("delta")
    .load(SILVER_RISK_PATH)
)

print("Rows:", check_risk.count())

print(
    "Unique zones:",
    check_risk
    .select("zone_id")
    .distinct()
    .count()
)

display(
    check_risk
    .groupBy("risk_category")
    .count()
    .orderBy("risk_category")
)

In [0]:
print("Rows:", len(zone_features))
print("Unique zones:", zone_features["zone_id"].nunique())

print(
    "Risk range:",
    zone_features["risk_score"].min(),
    "to",
    zone_features["risk_score"].max()
)

print(
    "Missing risk scores:",
    zone_features["risk_score"].isna().sum()
)

print(
    "Scores outside 0-100:",
    (~zone_features["risk_score"].between(0, 100)).sum()
)

print("\nRisk categories:")
print(zone_features["risk_category"].value_counts())

In [0]:
SILVER_RISK_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/current_risk"
)

risk_check = (
    spark.read
    .format("delta")
    .load(SILVER_RISK_PATH)
)

risk_check.select(
    "pipeline_run_timestamp"
).distinct().show(truncate=False)